In [1]:
import os
import numpy as np
import pandas as pd 
from pathlib import Path
from tqdm import tqdm

pd.options.mode.chained_assignment = None

DATA_PATH = Path('/data/gusev/USERS/jpconnor/data/')
CLINICAL_FEATURE_PATH = DATA_PATH / 'clinical_text_embedding_project/clinical_and_genomic_features/'
COMPASS_DATA_PATH = DATA_PATH / 'CAIA/COMPASS/'

ICD_df = pd.read_csv(COMPASS_DATA_PATH / 'prostate_icd_data.csv')
prostate_mrns = pd.read_csv(COMPASS_DATA_PATH / 'mrn_lists/longitudinal_mrns.csv')['DFCI_MRN']

NEPC_labels = pd.read_csv(COMPASS_DATA_PATH / 'LLM_NEPC_labels/LLM_v3_labels_just_NEPC.tsv', sep='\t')

NEPC_diagnosis_dates = (NEPC_labels.groupby('DFCI_MRN')['supporting_quote_date'].min()
                        .reset_index().rename(columns={'supporting_quote_date' : 'NEPC_diag_date'}))

# get most common ICDs to use in prediction model
n_ICDs = 50
ICD_cts = ICD_df[['DIAGNOSIS_ICD10_CD', 'DIAGNOSIS_ICD10_NM']].value_counts().reset_index()
most_common_ICDs = (ICD_cts.sort_values(by='count', ascending=False)['DIAGNOSIS_ICD10_CD'].tolist()[0:n_ICDs])

# prep ICD dataframe
ICD_df['START_DT'] = pd.to_datetime(ICD_df['START_DT'])
ICD_df = ICD_df.loc[ICD_df['DIAGNOSIS_ICD10_CD'].isin(most_common_ICDs),
                    ['DFCI_MRN', 'START_DT', 'DIAGNOSIS_ICD10_CD', 'DIAGNOSIS_ICD10_NM']]

ICD_tmp = ICD_df.copy()
ICD_tmp["START_DT"] = pd.to_datetime(ICD_tmp["START_DT"])

# first ICD date per MRN
ICD_tmp["first_date"] = ICD_tmp.groupby("DFCI_MRN")["START_DT"].transform("min")

# 30-day bins from first ICD date
ICD_tmp["monthly_bin"] = (
    (ICD_tmp["START_DT"] - ICD_tmp["first_date"]).dt.days // 30
)

# optional: keep only ICDs you care about before pivoting
ICD_tmp = ICD_tmp.loc[ICD_tmp["DIAGNOSIS_ICD10_CD"].isin(most_common_ICDs)]

# one-hot ICDs per MRN/month
full_NEPC_ICD_df = (
    ICD_tmp.assign(value=1)
    .pivot_table(
        index=["DFCI_MRN", "monthly_bin"],
        columns="DIAGNOSIS_ICD10_CD",
        values="value",
        aggfunc="max",
        fill_value=0
    )
    .reset_index()
)

# ensure all common ICD columns exist and ordered
full_NEPC_ICD_df = full_NEPC_ICD_df.reindex(
    columns=["DFCI_MRN", "monthly_bin"] + most_common_ICDs,
    fill_value=0
)

# NEPC month bin
first_dates = ICD_df.groupby("DFCI_MRN")["START_DT"].min()

nepc_bins = NEPC_diagnosis_dates.copy()
nepc_bins["NEPC_diag_date"] = pd.to_datetime(
    nepc_bins["NEPC_diag_date"].astype(str).str.strip(),
    errors="coerce"
)
nepc_bins["first_date"] = nepc_bins["DFCI_MRN"].map(first_dates)
nepc_bins["NEPC_bin"] = (
    (nepc_bins["NEPC_diag_date"] - nepc_bins["first_date"]).dt.days // 30
)

nepc_bins = nepc_bins[["DFCI_MRN", "NEPC_bin"]].dropna()
nepc_bins["NEPC"] = 1

# merge NEPC label onto MRN/month rows
full_NEPC_ICD_df = full_NEPC_ICD_df.merge(
    nepc_bins,
    left_on=["DFCI_MRN", "monthly_bin"],
    right_on=["DFCI_MRN", "NEPC_bin"],
    how="left"
)

full_NEPC_ICD_df["NEPC"] = full_NEPC_ICD_df["NEPC"].fillna(0).astype(int)

full_NEPC_ICD_df = full_NEPC_ICD_df[
    ["DFCI_MRN", "monthly_bin", "NEPC"] + most_common_ICDs
]

In [2]:
from scipy.stats import fisher_exact
import numpy as np
import pandas as pd

results = []

for icd in most_common_ICDs:

    a = ((full_NEPC_ICD_df[icd] == 1) &
         (full_NEPC_ICD_df["NEPC"] == 1)).sum()

    b = ((full_NEPC_ICD_df[icd] == 1) &
         (full_NEPC_ICD_df["NEPC"] == 0)).sum()

    c = ((full_NEPC_ICD_df[icd] == 0) &
         (full_NEPC_ICD_df["NEPC"] == 1)).sum()

    d = ((full_NEPC_ICD_df[icd] == 0) &
         (full_NEPC_ICD_df["NEPC"] == 0)).sum()

    # Haldane-Anscombe correction for zeros
    table = np.array([[a, b],
                      [c, d]], dtype=float)

    if (table == 0).any():
        table += 0.5

    or_val = (table[0,0] * table[1,1]) / (table[0,1] * table[1,0])

    _, p = fisher_exact([[a, b],
                         [c, d]])

    results.append({
        "ICD": icd,
        "OR": or_val,
        "p_value": p,
        "NEPC_with_ICD": a,
        "nonNEPC_with_ICD": b
    })

or_df = pd.DataFrame(results).sort_values(
    "OR", ascending=False
)

In [3]:
unique_codes = ICD_df[['DIAGNOSIS_ICD10_CD', 'DIAGNOSIS_ICD10_NM']].drop_duplicates()
code_to_name_map = dict(zip(unique_codes['DIAGNOSIS_ICD10_CD'], unique_codes['DIAGNOSIS_ICD10_NM']))

In [4]:
or_df['ICD_NM'] = or_df['ICD'].map(code_to_name_map)
or_df.loc[or_df['NEPC_with_ICD'] > 0, ['ICD', 'ICD_NM', 'NEPC_with_ICD', 'nonNEPC_with_ICD', 'OR']].iloc[0:20]

,ICD,ICD_NM,NEPC_with_ICD,nonNEPC_with_ICD,OR
24,Z51.5,Encounter for palliative care,12,875,14.280628
19,C80.1,"Malignant (primary) neoplasm, unspecified",5,335,14.250287
35,C67.9,"Malignant neoplasm of bladder, unspecified",6,543,10.657100
6,G89.3,Neoplasm related pain (acute) (chronic),14,1620,9.166380
48,R79.89,Other specified abnormal findings of blood che...,3,565,4.927566
0,C61,Malignant neoplasm of prostate,80,64352,4.332422
37,R31.9,"Hematuria, unspecified",2,601,3.048664
25,M54.50,"Low back pain, unspecified",2,626,2.925926
3,C79.51,Secondary malignant neoplasm of bone,25,11056,2.485427
30,Z85.46,Personal history of malignant neoplasm of pros...,3,1209,2.282816


### Predict using LR

In [5]:
# split data

from sklearn.model_selection import GroupShuffleSplit
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score, average_precision_score

# Data Prep
df = full_NEPC_ICD_df.copy()

feature_cols = most_common_ICDs
target_col = "NEPC"
group_col = "DFCI_MRN"

X = df[feature_cols].fillna(0).astype(int)
y = df[target_col].astype(int)
groups = df[group_col]

In [6]:
# Group train/test split
gss = GroupShuffleSplit(
    n_splits=1,
    test_size=0.2,
    random_state=42
)

train_idx, test_idx = next(gss.split(X, y, groups=groups))

X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]
groups_train = groups.iloc[train_idx]
groups_test = groups.iloc[test_idx]

print("Train NEPC rate:", y_train.mean())
print("Test NEPC rate:", y_test.mean())
print("Train MRNs:", groups_train.nunique())
print("Test MRNs:", groups_test.nunique())

Train NEPC rate: 0.001085848883246187
Test NEPC rate: 0.0011976844766784216
Train MRNs: 1540
Test MRNs: 385


In [7]:
X

,C61,Z79.899,Z51.11,C79.51,I10,E78.5,G89.3,I25.10,E78.00,E11.9,...,N52.9,Z98.890,L57.0,I26.99,N52.31,E78.2,G62.9,I63.9,R79.89,D69.6
0,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
1,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
2,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
3,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
4,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
74885,1,1,1,1,1,1,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
74886,1,1,1,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
74887,1,1,1,1,0,0,1,0,0,0,...,0,0,0,0,0,0,0,0,0,0
74888,1,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0


In [8]:
# Class imbalance weights
n_pos = y_train.sum()
n_neg = len(y_train) - n_pos

scale_pos_weight = n_neg / n_pos
print('scale_pos_weight:', scale_pos_weight)

sample_weight_train = np.where(
    y_train == 1, 
    scale_pos_weight,
    1.0
)

scale_pos_weight: 919.9384615384615


### LR Predictions

In [9]:
l1_logreg = Pipeline([
    ("scaler", StandardScaler(with_mean=False)),
    ("model", LogisticRegression(
        penalty="l1",
        solver="saga",
        class_weight="balanced",
        max_iter=5000,
        n_jobs=-1,
        random_state=42
    ))
])

l1_logreg.fit(X_train, y_train)

logreg_pred = l1_logreg.predict_proba(X_test)[:, 1]

print("L1 LogReg AUROC:", roc_auc_score(y_test, logreg_pred))
print("L1 LogReg AUPRC:", average_precision_score(y_test, logreg_pred))

/data/gusev/USERS/jpconnor/conda/envs/survlatent_ode/lib/python3.8/site-packages/sklearn/linear_model/_sag.py:350: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(


L1 LogReg AUROC: 0.632600907482661
L1 LogReg AUPRC: 0.002532877654245581


In [10]:
logreg_model = l1_logreg.named_steps["model"]

logreg_coef_df = pd.DataFrame({
    "ICD": feature_cols,
    "coef": logreg_model.coef_[0],
    "OR": np.exp(logreg_model.coef_[0])
})

logreg_coef_df = (
    logreg_coef_df
    .query("coef != 0")
    .sort_values("coef", ascending=False)
)

logreg_coef_df.head(30)

,ICD,coef,OR
0,C61,34.986445,1.564660e+15
37,R31.9,19.013009,1.808193e+08
36,E03.9,16.921449,2.233016e+07
46,G62.9,14.066426,1.285201e+06
24,Z51.5,13.085011,4.816682e+05
48,R79.89,12.357485,2.326954e+05
6,G89.3,10.214991,2.730952e+04
38,Z79.818,7.728691,2.272625e+03
33,R97.21,7.044343,1.146356e+03
18,Z51.81,6.179904,4.829456e+02


In [20]:
logreg_coef_df['ICD_NM'] = logreg_coef_df['ICD'].map(code_to_name_map)
logreg_coef_df['abs(coef)'] = np.abs(logreg_coef_df['coef'])
logreg_coef_df.sort_values(by='abs(coef)', ascending=False)[['ICD', 'ICD_NM', 'coef', 'abs(coef)']].head(15)

,ICD,ICD_NM,coef,abs(coef)
20,Z79.891,Long term (current) use of opiate analgesic,-72.960599,72.960599
14,G89.29,Other chronic pain,-68.798407,68.798407
31,G47.33,Obstructive sleep apnea (adult) (pediatric),-62.145449,62.145449
22,F41.9,"Anxiety disorder, unspecified",-59.530779,59.530779
15,K21.9,Gastro-esophageal reflux disease without esoph...,-55.942078,55.942078
13,Z00.6,Encounter for examination for normal compariso...,-54.013974,54.013974
28,F32.A,"Depression, unspecified",-51.980604,51.980604
49,D69.6,"Thrombocytopenia, unspecified",-49.468864,49.468864
26,M81.0,Age-related osteoporosis without current patho...,-48.636966,48.636966
44,N52.31,Erectile dysfunction following radical prostat...,-35.599325,35.599325


### XGBoost model

In [11]:
xgb = XGBClassifier(
    n_estimators=500,
    max_depth=3,
    learning_rate=0.03,
    subsample=0.8,
    colsample_bytree=0.8,
    min_child_weight=5,
    reg_lambda=1.0,
    objective="binary:logistic",
    eval_metric="aucpr",
    scale_pos_weight=scale_pos_weight,
    n_jobs=-1,
    random_state=42
)

xgb.fit(
    X_train,
    y_train,
    eval_set=[(X_test, y_test)],
    verbose=False
)

xgb_pred = xgb.predict_proba(X_test)[:, 1]

print("XGBoost AUROC:", roc_auc_score(y_test, xgb_pred))
print("XGBoost AUPRC:", average_precision_score(y_test, xgb_pred))

NameError: name 'XGBClassifier' is not defined

In [ ]:
xgb_importance_df = pd.DataFrame({
    "ICD": feature_cols,
    "importance": xgb.feature_importances_
}).sort_values("importance", ascending=False)

xgb_importance_df.head(30)